In [ ]:
from tabulate import tabulate
from astropy.coordinates import SkyCoord
from astropy.table import Table
import astropy.units as u
import numpy as np

In [ ]:
fits_path = '../output'

# Load asterisms
fits_file = f"{fits_path}/asterisms-GNAO-Optimal.fits"
#fits_file = f"{fits_path}/asterisms-GNAO-Nominal.fits"
#fits_file = f"{fits_path}/asterisms-GNAO-Limit.fits"
asterisms = Table.read(fits_file, format='fits')
print('Number of asterisms:', len(asterisms))

# Load targets
targets_name = 'sample-targets'
fits_file = f"{fits_path}/sample-targets.fits"
targets = Table.read(fits_file, format='fits')
col_formats = ["", ".0f", ".5f", ".5f", ".3f", ".1f", ".1f", ".1f"]

# targets = Table.read("../data/girmos-sci/lamiya-clusters.csv", format='csv')
# targets_name = 'lamiya-clusters'
# targets.rename_column('ID', 'id')
# targets.rename_column('Cluster', 'name')
# targets.rename_column('RA', 'ra')
# targets.rename_column('DEC', 'dec')
# col_formats = [".0f", "", "", "", ".3f"]

# targets_name = 'lammim_cluster_sample_redMapper_Kluge'
# targets = Table.read("../data/girmos-sci/lammim_cluster_sample_redMapper_Kluge.csv", format='csv')
# targets.rename_column('col0', 'id')
# targets.rename_column('z_cl', 'z')
# col_formats = [".0f", ".5f", ".5f", ".4f"]

# targets_name = 'lammim_cluster_sample_madcowsii'
# targets = Table.read("../data/girmos-sci/lammim_cluster_sample_madcowsii.csv", format='csv')
# targets.rename_column('col0', 'id')
# targets.rename_column('zspec', 'z')
# col_formats = [".0f", ".5f", ".5f", ".4f"]

# targets_name = 'lammim_cluster_sample_desi_legacy_wh24'
# targets = Table.read("../data/girmos-sci/lammim_cluster_sample_desi_legacy_wh24.csv", format='csv')
# targets.rename_column('col0', 'id')
# targets.rename_column('z_cl', 'z')
# col_formats = [".0f", ".5f", ".5f", ".4f"]

print('Number of targets:', len(targets))

In [ ]:
def get_asterism_code(star1_mag, star2_mag, star3_mag):
    bri_limit = 15.0
    nom_limit = 17.0
    dim_limit = 18.5

    codes = []
    for i in range(len(star1_mag)):
        num_bright = 0
        num_nominal = 0
        num_dim = 0

        if star1_mag[i] != -1:
            if star1_mag[i] < bri_limit:
                num_bright += 1
            elif star1_mag[i] < nom_limit:
                num_nominal += 1
            elif star1_mag[i] < dim_limit:
                num_dim += 1

        if star2_mag[i] != -1:
            if star2_mag[i] < bri_limit:
                num_bright += 1
            elif star2_mag[i] < nom_limit:
                num_nominal += 1
            elif star2_mag[i] < dim_limit:
                num_dim += 1

        if star3_mag[i] != -1:
            if star3_mag[i] < bri_limit:
                num_bright += 1
            elif star3_mag[i] < nom_limit:
                num_nominal += 1
            elif star3_mag[i] < dim_limit:
                num_dim += 1

        codes.append(f"{num_bright}{num_nominal}{num_dim}")
    
    return np.array(codes)

In [ ]:
if 'z' in targets.colnames:
    from astropy.constants import si as constants
    from survey_tools import sky

    R = 3000
    airmass = 1.5 # 1.0 or 1.5 or 2.0
    min_sky_rate = 10 # ph/s/m^2/arcsec^2/nm
    min_sky_trans = 0.8
    sky_trans_multiple = 0.5    # multiple of FWHM
    sky_line_multiple  = 0.5    # multiple of FWHM

    lines = [
        {'name': 'Ha'  , 'wavelength_vac': np.array([0.6564610           ]), 'sigma': 200},
        {'name': 'NII' , 'wavelength_vac': np.array([0.6549890, 0.6585270]), 'sigma': 200},
        {'name': 'Hb'  , 'wavelength_vac': np.array([0.4862680           ]), 'sigma': 200},
        {'name': 'OIII', 'wavelength_vac': np.array([0.4960295, 0.5008240]), 'sigma': 200},
    ]

    for l in lines:
        l['wavelength'] = sky.get_vacuum_to_air_wavelength(l['wavelength_vac']*u.micron).value
        targets[l['name']] = False
        col_formats.append(".0f")

    match R:
        case 3000:
            bands = [
                {'name': 'YJ', 'start': 0.95, 'end': 1.35},
                {'name': 'JH', 'start': 1.25, 'end': 1.80},
                {'name': 'HK', 'start': 1.63, 'end': 2.35},
            ]
        case 8000:
            bands = [
                {'name': 'Js', 'start': 1.194, 'end': 1.350},
                {'name': 'Hs', 'start': 1.500, 'end': 1.706},
                {'name': 'Ks', 'start': 2.110, 'end': 2.379},
            ]

    band_wavelength_range  = np.array([[b['start'], b['end']] for b in bands])
    band_wavelength_min = np.min(band_wavelength_range)
    band_wavelength_max = np.max(band_wavelength_range)
    print(f"Bands: {band_wavelength_min}-{band_wavelength_max} μm (R={R})")

    sky_transmission_data = sky.load_transmission_data('MaunaKea', airmass)
    print(f"Sky Transmission: {sky_transmission_data['wavelength'][0]/10:.0f}-{sky_transmission_data['wavelength'][-1]/10:.0f} nm, N={len(sky_transmission_data)}, dλ = {(sky_transmission_data['wavelength'][1]-sky_transmission_data['wavelength'][0])/10:.2f} nm")

    sky_background_data = sky.load_background_data('MaunaKea', airmass)
    print(f"Sky Background: {sky_background_data['wavelength'][0]/10:.0f}-{sky_background_data['wavelength'][-1]/10:.0f} nm, N={len(sky_background_data)}, dλ = {(sky_background_data['wavelength'][1]-sky_background_data['wavelength'][0])/10:.2f} nm")

    for t in targets:
        z = t['z']

        for l in lines:
            w = l['wavelength'] * (1 + z)
            fwhm = np.sqrt((w/R)**2 + (w * 2.35482 * l['sigma'] / constants.c.to('km/s').value)**2)

            reject = sky.reject_emission_line(
                sky_background_data,
                sky_transmission_data,
                w*1e4, # convert micron to angstrom
                fwhm*1e4, # convert micron to angstrom
                R,
                allowed_wavelength_range=band_wavelength_range*1e4, # convert micron to angstrom
                trans_minimum=min_sky_trans, 
                trans_dLambda_multiple=sky_trans_multiple, 
                avoid_dLambda_multiple=sky_line_multiple, 
                min_photon_rate=min_sky_rate
            )

            t[l['name']] = not np.all(reject)

In [ ]:
# Match targets to asterisms
asterism_catalog = SkyCoord(ra=asterisms['ra'], dec=asterisms['dec'], unit='deg', frame='icrs')
if isinstance(targets['ra'][0], str) and ':' in targets['ra'][0]:
    targets_catalog = SkyCoord(ra=targets['ra'], dec=targets['dec'], unit=(u.hourangle, u.deg), frame='icrs')
else:
    targets_catalog = SkyCoord(ra=targets['ra'], dec=targets['dec'], unit=(u.deg, u.deg), frame='icrs')
idx, sep, _ = targets_catalog.match_to_catalog_sky(asterism_catalog)
closest_asterism_id = asterisms['id'][idx]
closest_asterism_code = get_asterism_code(asterisms['star1_mag'][idx], asterisms['star2_mag'][idx], asterisms['star3_mag'][idx])

fov = 2*u.arcmin
target_filter = sep < fov/2

matches = targets[target_filter]
matches['asterism_id'] = closest_asterism_id[target_filter]
matches['asterism_code'] = closest_asterism_code[target_filter]
col_formats = col_formats + [".0f", ".0f"]

if 'field' in matches.colnames:
    matches.sort(['field', 'id'])
else:
    matches.sort(['id'])

print('Number of targets near an optimal asterism:', len(matches))
display(tabulate(matches, headers=matches.colnames, tablefmt='html', 
                 floatfmt=col_formats))

matches.write(f"../output/matches-{targets_name}.csv", format='csv', overwrite=True)